# Notebook 01 — JIT Data Scraping

**Student:** Rohit Sharma | 19385207

This notebook collects a just-in-time (JIT) dataset of live Google Play Store reviews using the `google-play-scraper` library.

---

## Step 1: Install dependencies

In [1]:
# Run this cell first (only needed once in Colab)
!pip install google-play-scraper pandas

zsh:1: command not found: pip


## Step 2: Import libraries

In [2]:
from google_play_scraper import reviews, Sort
import pandas as pd
import time
import os

print('Libraries imported successfully')

Libraries imported successfully


## Step 3: Define apps to scrape

We scrape from a variety of app categories to ensure diversity in our dataset.

In [3]:
# App IDs from Google Play Store — covering different categories
APPS = {
    'productivity': [
        'com.todoist',           # Todoist
        'com.microsoft.teams',   # Microsoft Teams
        'com.microsoft.todos',         # Microsoft To Do
    ],
    'entertainment': [
        'com.netflix.mediaclient',  # Netflix
        'com.spotify.music',        # Spotify
    ],
    'health': [
        'com.myfitnesspal.android',  # MyFitnessPal
        'com.calm.android',     # Calm
    ],
    'social': [
        'com.instagram.android',  # Instagram
        'org.telegram.messenger',           # Telegram
    ],
    'finance': [
        'com.paypal.android.p2pmobile',  # PayPal
        'com.revolut.revolut',             # Revolut
    ]
}

print(f'Total apps to scrape: {sum(len(v) for v in APPS.values())}')

Total apps to scrape: 11


## Step 4: Scrape reviews

In [4]:
def scrape_app_reviews(app_id, category, count=200):
    """
    Scrape reviews for a single app.
    Returns a list of review dictionaries.
    """
    try:
        result, _ = reviews(
            app_id,
            lang='en',
            country='us',
            sort=Sort.NEWEST,
            count=count
        )
        for r in result:
            r['app_id'] = app_id
            r['category'] = category
        print(f'  Scraped {len(result)} reviews for {app_id}')
        return result
    except Exception as e:
        print(f'  ERROR scraping {app_id}: {e}')
        return []


all_reviews = []

for category, app_list in APPS.items():
    print(f'\nScraping category: {category}')
    for app_id in app_list:
        reviews_data = scrape_app_reviews(app_id, category, count=300)
        all_reviews.extend(reviews_data)
        time.sleep(2)  # Be polite — avoid rate limiting

print(f'\nTotal reviews collected: {len(all_reviews)}')


Scraping category: productivity
  Scraped 300 reviews for com.todoist
  Scraped 300 reviews for com.microsoft.teams
  Scraped 300 reviews for com.microsoft.todos

Scraping category: entertainment
  Scraped 300 reviews for com.netflix.mediaclient
  Scraped 300 reviews for com.spotify.music

Scraping category: health
  Scraped 300 reviews for com.myfitnesspal.android
  Scraped 300 reviews for com.calm.android

Scraping category: social
  Scraped 300 reviews for com.instagram.android
  Scraped 300 reviews for org.telegram.messenger

Scraping category: finance
  Scraped 300 reviews for com.paypal.android.p2pmobile
  Scraped 300 reviews for com.revolut.revolut

Total reviews collected: 3300


## Step 5: Convert to DataFrame and inspect

In [5]:
df = pd.DataFrame(all_reviews)

# Keep only the columns we need
df = df[['reviewId', 'content', 'score', 'at', 'app_id', 'category']]

# Rename for clarity
df.columns = ['review_id', 'review_text', 'star_rating', 'date', 'app_id', 'app_category']

print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nStar rating distribution:')
print(df['star_rating'].value_counts().sort_index())
df.head(10)

Dataset shape: (3300, 6)

Columns: ['review_id', 'review_text', 'star_rating', 'date', 'app_id', 'app_category']

Star rating distribution:
star_rating
1     931
2     205
3     179
4     242
5    1743
Name: count, dtype: int64


,review_id,review_text,star_rating,date,app_id,app_category
0,67637fcd-ad48-443c-abc5-c1f27d4bb4c7,How I wish I could give more stars. Love this ...,3,2026-07-22 00:20:36,com.todoist,productivity
1,67f2452e-6fc2-4c96-b73a-8d27c875f9f4,very useful,5,2026-07-21 13:45:26,com.todoist,productivity
2,01aa7769-f11b-49a6-901c-3b18cee070b1,Mediocre,3,2026-07-21 13:33:58,com.todoist,productivity
3,fcfc6d32-0259-4370-bf27-f1b5ec64b25b,Time issue what if a person sleeps 5am or 6am....,1,2026-07-20 20:24:51,com.todoist,productivity
4,dafa0c81-b358-4c97-bbfe-a2f0b8b8d3c1,I downloaded this app and found that it did no...,1,2026-07-20 16:03:35,com.todoist,productivity
5,ea0b4c48-d053-4fff-a551-486836daa18c,good,4,2026-07-20 01:57:17,com.todoist,productivity
6,5ba1dcea-6088-4121-9c9a-ce797e2788f4,works exactly as described. too bad that the p...,5,2026-07-20 00:50:11,com.todoist,productivity
7,2e6146ce-2711-4ee0-8dfb-95aced4d6aa0,they just added AI features. dont use,1,2026-07-19 15:20:11,com.todoist,productivity
8,5c0a5283-dc73-424f-b976-872b5036c852,I've been avoiding getting this app because I ...,5,2026-07-19 08:31:42,com.todoist,productivity
9,0aa18638-1d1d-4e06-9f5f-f863e71e1e5a,There a snackbar whenever I reorder my task li...,4,2026-07-19 02:52:31,com.todoist,productivity


## Step 6: Basic cleaning

In [6]:
# Remove duplicates
before = len(df)
df = df.drop_duplicates(subset='review_text')
print(f'Removed {before - len(df)} duplicate reviews')

# Remove empty reviews
df = df.dropna(subset=['review_text'])
df = df[df['review_text'].str.strip() != '']

# Remove very short reviews (less than 5 words) — not useful for classification
df = df[df['review_text'].str.split().str.len() >= 5]

print(f'Final dataset size: {len(df)} reviews')
df.head()

Removed 360 duplicate reviews
Final dataset size: 2090 reviews


,review_id,review_text,star_rating,date,app_id,app_category
0,67637fcd-ad48-443c-abc5-c1f27d4bb4c7,How I wish I could give more stars. Love this ...,3,2026-07-22 00:20:36,com.todoist,productivity
3,fcfc6d32-0259-4370-bf27-f1b5ec64b25b,Time issue what if a person sleeps 5am or 6am....,1,2026-07-20 20:24:51,com.todoist,productivity
4,dafa0c81-b358-4c97-bbfe-a2f0b8b8d3c1,I downloaded this app and found that it did no...,1,2026-07-20 16:03:35,com.todoist,productivity
6,5ba1dcea-6088-4121-9c9a-ce797e2788f4,works exactly as described. too bad that the p...,5,2026-07-20 00:50:11,com.todoist,productivity
7,2e6146ce-2711-4ee0-8dfb-95aced4d6aa0,they just added AI features. dont use,1,2026-07-19 15:20:11,com.todoist,productivity


## Step 7: Save raw dataset

In [7]:
# Save to CSV
os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/reviews_raw.csv', index=False)
print(f'Saved {len(df)} reviews to ../data/raw/reviews_raw.csv')


Saved 2090 reviews to ../data/raw/reviews_raw.csv


---
## Summary

| Item | Value |
|------|-------|
| Total reviews collected | See output above |
| Apps scraped | 10 apps across 5 categories |
| Language | English only |
| Output file | `data/raw/reviews_raw.csv` |
